# M19c - Test-time prediction-variation indicators

**Author:** Ildefons Magrans de Abril  
**Affiliation:** Universitat Politècnica de Catalunya - BarcelonaTech (UPC)

**Submission evidence.** Window-level indicator associations and cluster-bootstrap uncertainty are regenerated from the current executable protocol.

In [1]:
from pathlib import Path
import sys, numpy as np, pandas as pd
from IPython.display import display
ROOT=Path.cwd()
if not (ROOT/'src').exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'src'))
import tcr_core as tcr
REPRO=ROOT/'results'/'reproduced'; REPRO.mkdir(parents=True,exist_ok=True)

In [2]:
def cluster_quartile_boot(df,indicator,n_boot=30000,seed=1,batch=250):
    ids=list(df.case_id.drop_duplicates()); I=[]; G=[]
    for cid in ids:
        z=df[df.case_id==cid]; I.append(z[indicator].to_numpy(float)); G.append(z.safe_gain.to_numpy(float))
    I=np.stack(I); G=np.stack(G); C,W=I.shape
    iv=I.ravel(); gv=G.ravel(); ok=np.isfinite(iv); iv=iv[ok]; gv=gv[ok]
    q1,q3=np.quantile(iv,[.25,.75]); point=float(gv[iv>=q3].mean()-gv[iv<=q1].mean())
    rng=np.random.default_rng(seed); out=np.empty(n_boot); pos=0
    while pos<n_boot:
        b=min(batch,n_boot-pos); choices=rng.integers(0,C,size=(b,C))
        Is=I[choices].reshape(b,C*W); Gs=G[choices].reshape(b,C*W)
        q1s=np.nanquantile(Is,.25,axis=1); q3s=np.nanquantile(Is,.75,axis=1)
        hi=np.where(Is>=q3s[:,None],Gs,np.nan); lo=np.where(Is<=q1s[:,None],Gs,np.nan)
        out[pos:pos+b]=np.nanmean(hi,axis=1)-np.nanmean(lo,axis=1); pos+=b
    return point,float(np.quantile(out,.025)),float(np.quantile(out,.975))

In [3]:
SEED=20260623; TRIALS=12
TASKS=['controlled_d20_white_plus_distractor','memory_d10','narma10','lorenz_x']
CONFIG=dict(N=60,K=13,lengths=(1200,500,500),washout=100,input_scale=.8,ridge=1e-5)
parts=[]
for task in TASKS:
    for trial in range(TRIALS):
        case=tcr.evaluate_case(task,trial,'temperature',SEED,return_predictions=True,**CONFIG)
        w=tcr.window_rows(case,window_size=50,stride=25); w['task']=task; w['trial']=trial; w['case_id']=f'{task}__{trial}'; parts.append(w)
windows=pd.concat(parts,ignore_index=True); windows.to_csv(REPRO/'m19c_replication_window_metrics.csv',index=False)
d=cluster_quartile_boot(windows,'safe_dispersion',seed=1904); a=cluster_quartile_boot(windows,'anchor_spread',seed=1905)
summary=pd.DataFrame([{'n_windows':len(windows),'safe_dispersion_gain_corr':tcr.safe_corr(windows.safe_dispersion,windows.safe_gain),'anchor_spread_gain_corr':tcr.safe_corr(windows.anchor_spread,windows.safe_gain),'local_sensitivity_gain_corr':tcr.safe_corr(windows.local_sensitivity,windows.safe_gain),'safe_dispersion_high_minus_low_gain':d[0],'safe_dispersion_ci_low':d[1],'safe_dispersion_ci_high':d[2],'anchor_spread_high_minus_low_gain':a[0],'anchor_spread_ci_low':a[1],'anchor_spread_ci_high':a[2]}])
summary.to_csv(REPRO/'m19c_replication_summary.csv',index=False)
display(summary.round(6))

,n_windows,safe_dispersion_gain_corr,anchor_spread_gain_corr,local_sensitivity_gain_corr,safe_dispersion_high_minus_low_gain,safe_dispersion_ci_low,safe_dispersion_ci_high,anchor_spread_high_minus_low_gain,anchor_spread_ci_low,anchor_spread_ci_high
0,912,0.322929,0.387012,0.327759,0.017544,0.0076,0.031107,0.020626,0.008511,0.033696
